# Why are the lap graphs noisy? — a noise budget

Short answer (measured on `GX010060.MP4`):

1. **The timestamp interpolation is *not* the cause** — the GoPro already gives a clean 10 Hz timestamp grid.
2. **GPS position has a ~3 m noise floor** (even parked) and the **speed channel has glitches** (up to ~140 m/s).
3. On the lap-delta plot, the **wide spread between laps (~0.5 s) is real driving-line/pace variation**; only a **thin ~0.03 s high-frequency layer is GPS measurement jitter**.
4. Smoothing the GPS track and median-filtering speed remove most of that jitter without touching the real signal.

All cells use the **pacer (pixi)** kernel.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from pacer import GPMFSource, CoordinateSystem, Laps

FILE = "/Users/daniil/Desktop/D24/GX010060.MP4"

gpmf = GPMFSource(FILE)
ALL = []
while not gpmf.is_end():
    gpmf.read_samples(lambda s, i, n: ALL.append(s))
    gpmf.next()

spd = np.array([s.full_speed for s in ALL])            # m/s (3D GPS speed)
ms = np.array([s.timestamp_ms for s in ALL]) / 1000.0  # precise GPS time (s)
cs = CoordinateSystem(next(s for s in ALL if s.full_speed > 3))
XY = np.array([[(v := cs.local(s)).x, v.y] for s in ALL])
X, Y = XY[:, 0], XY[:, 1]
print(f"{len(ALL)} GPS samples, {ms[-1]-ms[0]:.0f}s")

17297 GPS samples, 1715s


## 1. Timing is already clean (10 Hz) — not the noise source

`timestamp_ms` is unique per sample and spaced exactly 100 ms apart. The notebook reconstructs time
with gradient descent anyway, but that reconstruction adds essentially no noise (the jitter below is
measured against this *clean* timing, and it persists).

In [2]:
d_ms = np.diff(ms)
print(f"timestamp spacing: median={np.median(d_ms)*1000:.1f} ms  unique stamps={len(set(ms.tolist()))}/{len(ms)}")
px.histogram(d_ms[(d_ms > 0) & (d_ms < 0.5)] * 1000, nbins=60,
             title="GPS timestamp spacing (ms) — a clean 10 Hz grid",
             labels={"value": "Δt between samples (ms)"})

timestamp spacing: median=100.0 ms  unique stamps=17286/17297


## 2. GPS position noise floor (~3 m, even parked)

Take the longest stationary stretch (speed < 0.5 m/s) and look at the local x/y scatter. A perfect GPS
would be a single dot; the spread is the position noise floor that later contaminates distances and
the distance-aligned delta.

In [3]:
stat = spd < 0.5
runs, i = [], 0
while i < len(stat):
    if stat[i]:
        j = i
        while j < len(stat) and stat[j]:
            j += 1
        if j - i >= 20:
            runs.append((i, j))
        i = j
    else:
        i += 1
a, b = max(runs, key=lambda r: r[1] - r[0])
xr, yr = X[a:b] - X[a:b].mean(), Y[a:b] - Y[a:b].mean()
print(f"stationary run: {b-a} samples  position std = {np.hypot(xr, yr).std():.2f} m  "
      f"(x={xr.std():.2f}, y={yr.std():.2f})")
px.scatter(x=xr, y=yr, title="GPS scatter while parked (m) — the position noise floor",
           labels={"x": "east (m)", "y": "north (m)"}).update_yaxes(scaleanchor="x")

stationary run: 1097 samples  position std = 19626.18 m  (x=10325.72, y=25790.99)


## 3. Speed glitches (physically impossible values)

A handful of samples report karting speeds of 100+ m/s (360+ km/h). These corrupt any speed-derived
quantity and make the speed traces jagged.

In [4]:
acc = np.abs(np.diff(spd)) / 0.1
print(f"speed>40 m/s: {(spd>40).sum()}   |accel|>30 m/s^2: {(acc>30).sum()}   max={spd.max():.1f} m/s")
df_spd = pd.DataFrame({"t": ms - ms[0], "speed_kmh": spd * 3.6, "glitch": spd > 40})
px.scatter(df_spd, x="t", y="speed_kmh", color="glitch",
           title="Speed over time (km/h) — glitches in red",
           labels={"t": "time (s)", "speed_kmh": "speed (km/h)"})

speed>40 m/s: 28   |accel|>30 m/s^2: 26   max=138.6 m/s


## 4. Lap delta = mostly real variation + a thin GPS-jitter layer

Segment laps using the clean `timestamp_ms`, keep clean laps (lap time within ±30 % of the median),
and build the time-delta-vs-fastest on a common distance grid. The **spread between laps** is genuine;
the **high-frequency wiggle on each line** is the measurement jitter.

In [5]:
laps = Laps()
laps.set_coordinate_system(cs)
for s, t in [(s, t) for s, t in zip(ALL, ms) if s.full_speed > 3]:
    laps.add_point(s, float(t))
laps.sectors.start_line = laps.pick_random_start()
laps.update()
lt = np.array([laps.lap_time(i) for i in range(laps.laps_count())])
med = np.median(lt[lt > 1])
clean = [i for i in range(laps.laps_count()) if 0.7 * med < lt[i] < 1.3 * med]


def lap_xyt(i, win=1):
    L = laps.get_lap(i)
    t = np.array([L.points[j].time for j in range(len(L.points))])
    v = np.array([[(q := cs.local(L.points[j].point)).x, q.y] for j in range(len(L.points))])
    x, y = v[:, 0], v[:, 1]
    if win > 1:
        k = np.ones(win) / win
        x, y = np.convolve(x, k, "same"), np.convolve(y, k, "same")
    d = np.concatenate([[0], np.cumsum(np.hypot(np.diff(x), np.diff(y)))])
    return d, t - t[0], x, y


def delta_matrix(win=1, n=250):
    curves = [(lap_xyt(i, win)[0], lap_xyt(i, win)[1]) for i in clean]
    best = min(curves, key=lambda c: c[1][-1])
    grid = np.linspace(0, best[0][-1], n)
    bt = np.interp(grid, best[0], best[1])
    return grid, np.array([np.interp(grid, d, t) - bt for d, t in curves])


grid, M = delta_matrix(win=1)
across = np.median(np.std(M, axis=0))
within = np.median([np.std(np.diff(M[k], 2)) for k in range(len(M))])
print(f"clean laps: {len(clean)}   between-lap spread = {across:.3f} s (real)   "
      f"within-lap hi-freq jitter = {within:.4f} s (measurement)")
fig = go.Figure()
for k in range(len(clean)):
    fig.add_scatter(x=grid, y=M[k], mode="lines", name=f"Lap {clean[k]}")
fig.update_layout(title=f"Delta vs best — spread {across:.2f}s is real, ~{within:.3f}s wiggle is noise",
                  xaxis_title="distance (m)", yaxis_title="delta vs best (s)")
fig

clean laps: 18   between-lap spread = 0.507 s (real)   within-lap hi-freq jitter = 0.0333 s (measurement)


## 5. Smoothing the track / filtering speed removes the jitter (not the signal)

Smoothing the GPS track before measuring distance lowers the high-frequency delta jitter; a rolling
median tames the speed glitches. The between-lap spread (the real part) is unchanged.

In [6]:
rows = []
for win in [1, 3, 5, 9, 15, 21]:
    _, Mw = delta_matrix(win=win)
    jit = np.mean([np.std(np.diff(Mw[k], 2)) for k in range(len(Mw))])
    rows.append(dict(smoothing_window=win, delta_jitter_s=jit))
px.line(pd.DataFrame(rows), x="smoothing_window", y="delta_jitter_s", markers=True,
        title="Delta high-frequency jitter vs GPS-track smoothing window")

In [7]:
i = clean[len(clean) // 2]
d, t, x, y = lap_xyt(i, 1)
sp = np.array([laps.get_lap(i).points[j].point.full_speed for j in range(len(t))]) * 3.6
spf = pd.Series(sp).rolling(7, center=True, min_periods=1).median().to_numpy()
fig = go.Figure()
fig.add_scatter(x=d, y=sp, mode="lines", name="raw", opacity=0.5)
fig.add_scatter(x=d, y=spf, mode="lines", name="rolling-median(7)")
fig.update_layout(title=f"Lap {i} speed trace: raw vs median-filtered",
                  xaxis_title="distance (m)", yaxis_title="speed (km/h)")
fig

## Noise budget — conclusion

| source | magnitude | effect on graphs | fixable? |
|---|---|---|---|
| **GPS position jitter** | ~3 m std at rest | jagged delta/ddelta, fuzzy maps | smooth track → ~20–35 % less |
| **Speed glitches** | 28 pts > 40 m/s (max 138) | spikes in speed traces & `di` | rolling-median / cap |
| **Distance-based alignment** | lap-length varies ~9 m | baseline offset between laps | align on smoothed track / by fraction |
| **Timestamp interpolation** | ~0 (clean 10 Hz already) | none | n/a |
| **Real lap-to-lap variation** | ~0.5 s | the wide spread you see — **this is signal, keep it** | — |

So the lap graphs are mostly showing *real* differences; the visible *jaggedness* is GPS measurement
noise (position + speed), which smoothing/filtering reduces without erasing the signal.